In [1]:
import torch
import torch.nn as nn

In [5]:
torch.tril(torch.ones(5, 5))

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

In [6]:
class MultiQueryAttentionSrc(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0, \
            "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, self.d_head)
        self.W_v = nn.Linear(d_model, self.d_head)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(1, 1, 1024, 1024), diagonal=1))

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        k = self.W_k(x).view(batch_size, seq_len, 1, self.d_head).transpose(1, 2)
        v = self.W_v(x).view(batch_size, seq_len, 1, self.d_head).transpose(1, 2)

        k = k.repeat(1, self.num_heads, 1, 1)
        v = v.repeat(1, self.num_heads, 1, 1)

        attn_scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)

        attn_scores = attn_scores.masked_fill(self.mask[:,:,:seq_len,:seq_len] == 0, float('-inf'))

        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vector = (attn_weights @ v).transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        output = self.W_o(context_vector)
        return output

In [10]:
class MultiQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1 ):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.dropout_rate = dropout

        self.WQ = nn.Linear(d_model, d_model)
        self.WK = nn.Linear(d_model, self.head_dim)
        self.WV = nn.Linear(d_model, self.head_dim)
        self.WO = nn.Linear(d_model, d_model)

        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(self.dropout_rate)
        self.register_buffer('mask', torch.tril(torch.ones(1, 1, 1024, 1024)))

    def forward(self, X):
        batch_size, seq_len, _ = X.shape

        Q = self.WQ(X)
        K = self.WK(X)
        V = self.WV(X)
        Q = Q.reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.reshape(batch_size, seq_len, 1, self.head_dim).transpose(1, 2)
        V = V.reshape(batch_size, seq_len, 1, self.head_dim).transpose(1, 2)
        K = K.repeat(1, self.num_heads, 1, 1)
        V = V.repeat(1, self.num_heads, 1, 1)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_scores = attn_scores.masked_fill(self.mask[:, :, :seq_len, :seq_len] == 0, (-1) * torch.inf)
        attn_weights = self.softmax(attn_scores)
        context = torch.matmul(attn_weights, V)
        #  context = self.dropout(context)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.WO(context)

        return output

In [11]:
mqa = MultiQueryAttention(16, 4, 0.1)
X = torch.randn(4,15,16)

In [13]:
mqa(X).shape

torch.Size([4, 15, 16])